Performance 🏃 
---
https://github.com/jeffmur/fhe-video-similarity/wiki/experiment

This notebook serves to analyze, compare, and visulize the trade-off of performance when using Fully Homomorphic Encryption (FHE) to compute video similarity scores on mobile vs. desktop devices.

Note: The multi-threading feature for pre-processing is disabled for all experiments, as we value consistency over performance.

Note: Every experiment uses the same encryption scheme & parameters:

* Cryptosystem: CKKS
* Polynomial Degree: 4096
* Encode Scalar: 2^40
* qSizes: [60, 40, 40, 60]

There are three metrics being gather within the application:

⚙️ **Pre-processing Time**: The time it takes to convert the video into a format that can be used for comparison.

📊 **Similarity Scores**: The time it takes to encrypt & compute a similarity score.

# Experiment 1: On-device Comparison

In this experiment, we compare the pre-processing and encryption duration on mobile vs. desktop devices.

We aim to learn how the performance of the application varies amongst devices (mobile vs. desktop). The scenarios are split by resolution (720p, 1080p, 2160p) and the video length is a constant 60 seconds. We select 60 seconds, as there is a direct comparison between the devices, as well as baseline comparison to Pop-Share.

| Device | OS
| --- | --- |
| Samsung S9 | Android
| Pixel 3XL | Android
| PC | Linux
| Raspberry Pi 400 | Linux

In [1]:
from utils.performance_tables import *
from utils import TARGET_SYS, FRAME_COUNTS
# To be summarized in table
kld_err = []
bhatt_err = []
cram_err = []

def add_mean_error(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    kld, bhattacharyya, cramer = mean_error(pathToAssertion, os, frameCounts).values()
    kld_err.append(kld)
    bhatt_err.append(bhattacharyya)
    cram_err.append(cramer)

# Aggregated Pre-processing durations
pp_by_sys = {}
pp_by_res = {}

# Operations FHE vs. Plaintext
ops_by_sys = {}
ops_by_sys_alg = {}
mobile_ops_by_alg = {}

def add_metric(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    # OS pre-processing
    insert_or_append(pp_by_sys, pre_processing_by_sys(pathToAssertion, os, frameCounts))
    
    # Resolution pre-processing
    res = pre_processing_by_res(pathToAssertion.split('/')[1], pathToAssertion, os, frameCounts)
    insert_or_append(pp_by_res, res)
    
    # Mean Error FHE vs. Plaintext
    add_mean_error(pathToAssertion, os, frameCounts)

    # Operations FHE vs. Plaintext
    insert_or_append(ops_by_sys, operations_by_sys(pathToAssertion, os, frameCounts))

    # Operations FHE vs. Plaintext by Algorithm
    insert_or_append(mobile_ops_by_alg, operations_by_alg(pathToAssertion, ['pxl', 's9'], frameCounts))

    # Operations FHE vs. Plaintext by Algorithm & System
    insert_or_append(ops_by_sys_alg, operations_by_sys_alg(pathToAssertion, os, frameCounts))

## Scenario 1: 720p

On every device, import and pre-process the video twice, compare against itself, using two different keys.


### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1280x720 -c:v libx264 -t 60 -an Black_720p_60s.mp4
```

In [2]:
black_720p = "3_performance/720p/60s_Black"
add_metric(black_720p)
verbose_md_table(black_720p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | Decryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---|---
KLD [pc] [all] | 1.10e-11 [100.00%] | 2.20e-11 | 13.00 | 151.00 | 32.00 | 51.00 | 0.34 | 50.66 [14812.28%]
Cramer [pc] [all] | 1.08e-09 [100.00%] | 2.16e-09 | 13.00 | 73.00 | 31.00 | 45.00 | 0.10 | 44.90 [46291.75%]
BC [pc] [all] | 1.00e+00 [100.00%] | 4.19e-10 | 13.00 | 73.00 | 23.00 | 32.00 | 0.08 | 31.92 [41458.44%]
KLD [s9] [all] | 2.39e-11 [100.00%] | 4.78e-11 | 180.00 | 971.00 | 207.00 | 311.00 | 1.00 | 310.00 [31000.00%]
Cramer [s9] [all] | 1.68e-10 [100.00%] | 3.36e-10 | 180.00 | 501.00 | 207.00 | 289.00 | 0.50 | 288.50 [57584.63%]
BC [s9] [all] | 1.00e+00 [100.00%] | 5.10e-11 | 180.00 | 504.00 | 137.00 | 198.00 | 0.39 | 197.61 [50539.39%]
KLD [pxl] [all] | 1.83e-12 [100.00%] | 3.66e-12 | 171.00 | 961.00 | 201.00 | 310.00 | 2.00 | 308.00 [15400.00%]
Cramer [pxl] [all] | 4.03e-10 [100.00%] | 8.06e-10 | 171.00 | 439.00 | 193.00 | 281.00 | 0.33 | 280.67 [85310.33%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 1.73e-12 | 171.00 | 444.00 | 122.00 | 179.00 | 0.33 | 178.67 [54142.42%]

### Test 2: Samsung S9

Taken from Handheld Samsung S9, 720p, 60 seconds.

In [3]:
s9_720p = "3_performance/720p/60s_S9"
add_metric(s9_720p)
verbose_md_table(s9_720p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | Decryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---|---
KLD [pc] [all] | 3.32e-12 [100.00%] | 6.64e-12 | 74.00 | 157.00 | 33.00 | 53.00 | 0.34 | 52.66 [15720.90%]
Cramer [pc] [all] | 3.64e-11 [100.00%] | 7.29e-11 | 74.00 | 75.00 | 31.00 | 45.00 | 0.09 | 44.91 [49900.00%]
BC [pc] [all] | 1.00e+00 [100.00%] | 4.44e-11 | 74.00 | 76.00 | 24.00 | 33.00 | 0.08 | 32.92 [41672.15%]
KLD [s9] [all] | 1.95e-11 [100.00%] | 3.89e-11 | 416.50 | 883.00 | 195.00 | 294.00 | 1.00 | 293.00 [29300.00%]
Cramer [s9] [all] | 8.88e-10 [100.00%] | 1.78e-09 | 416.50 | 438.00 | 191.00 | 269.00 | 0.48 | 268.52 [55593.58%]
BC [s9] [all] | 1.00e+00 [100.00%] | 3.23e-10 | 416.50 | 427.00 | 120.00 | 168.00 | 0.36 | 167.64 [46958.82%]
KLD [pxl] [all] | 7.14e-11 [100.00%] | 1.43e-10 | 418.50 | 976.00 | 233.00 | 348.00 | 1.00 | 347.00 [34700.00%]
Cramer [pxl] [all] | 2.13e-09 [100.00%] | 4.25e-09 | 418.50 | 484.00 | 212.00 | 309.00 | 0.33 | 308.67 [94685.28%]
BC [pxl] [all] | 1.00e+00 [100.00%] | 2.90e-10 | 418.50 | 481.00 | 134.00 | 198.00 | 0.30 | 197.70 [65031.58%]

## Scenario 2: 1080p



### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1920x1080 -c:v libx264 -t 60 -an Black_1080p_60s.mp4
```

In [4]:
black_1080p = "3_performance/1080p/60s_Black"
add_metric(black_1080p)
verbose_md_table(black_1080p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | Decryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---|---
KLD [pc] [all] | 7.32e-11 [100.00%] | 1.46e-10 | 37.00 | 146.00 | 30.00 | 44.00 | 0.03 | 43.97 [133233.33%]
Cramer [pc] [all] | 1.21e-09 [100.00%] | 2.42e-09 | 37.00 | 72.00 | 30.00 | 42.00 | 0.02 | 41.98 [199900.00%]
BC [pc] [all] | 1.00e+00 [100.00%] | 7.84e-10 | 37.00 | 73.00 | 23.00 | 29.00 | 0.02 | 28.98 [181150.00%]
KLD [s9] [all] | 5.26e-11 [100.00%] | 1.05e-10 | 334.50 | 972.00 | 217.00 | 335.00 | 1.00 | 334.00 [33400.00%]
Cramer [s9] [all] | 1.19e-09 [100.00%] | 2.38e-09 | 334.50 | 516.00 | 211.00 | 297.00 | 0.52 | 296.48 [56687.76%]
BC [s9] [all] | 1.00e+00 [100.00%] | 7.99e-10 | 334.50 | 497.00 | 145.00 | 207.00 | 0.59 | 206.41 [34984.75%]
KLD [pxl] [all] | 3.76e-07 [100.00%] | 1.67e-12 | 353.50 | 1215.00 | 284.00 | 420.00 | 1.00 | 419.00 [41900.00%]
Cramer [pxl] [all] | 2.22e-04 [100.00%] | 2.81e-04 | 353.50 | 593.00 | 259.00 | 370.00 | 0.46 | 369.54 [80334.78%]
BC [pxl] [all] | 1.00e+00 [99.98%] | 9.01e-11 | 353.50 | 593.00 | 165.00 | 241.00 | 0.53 | 240.47 [45804.76%]

### Test 2: Pixel 3XL

Taken from Handheld Pixel 3XL, 1080p, 60 seconds.

In [5]:
pxl_1080p = "3_performance/1080p/60s_PXL"
add_metric(pxl_1080p)
verbose_md_table(pxl_1080p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | Decryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---|---
KLD [pc] [all] | 1.50e-11 [100.00%] | 3.00e-11 | 321.00 | 155.00 | 31.00 | 48.00 | 0.33 | 47.67 [14445.45%]
Cramer [pc] [all] | 8.92e-10 [100.00%] | 1.78e-09 | 321.00 | 74.00 | 31.00 | 43.00 | 0.10 | 42.90 [44691.67%]
BC [pc] [all] | 1.00e+00 [100.00%] | 2.92e-10 | 321.00 | 73.00 | 24.00 | 30.00 | 0.07 | 29.93 [40440.54%]
KLD [s9] [all] | 4.37e-11 [100.00%] | 8.74e-11 | 755.50 | 975.00 | 220.00 | 323.00 | 1.00 | 322.00 [32200.00%]
Cramer [s9] [all] | 1.50e-09 [100.00%] | 3.01e-09 | 755.50 | 492.00 | 208.00 | 291.00 | 0.54 | 290.46 [53788.89%]
BC [s9] [all] | 1.00e+00 [100.00%] | 9.15e-10 | 755.50 | 484.00 | 133.00 | 187.00 | 0.36 | 186.63 [51132.88%]
KLD [pxl] [all] | 1.27e-05 [100.00%] | 8.12e-11 | 875.00 | 1199.00 | 282.00 | 422.00 | 1.00 | 421.00 [42100.00%]
Cramer [pxl] [all] | 1.55e-03 [100.00%] | 1.95e-03 | 875.00 | 594.00 | 259.00 | 369.00 | 0.44 | 368.56 [83008.11%]
BC [pxl] [all] | 1.00e+00 [99.85%] | 9.81e-11 | 875.00 | 594.00 | 166.00 | 241.00 | 0.38 | 240.62 [63656.61%]

# Experiment 2: FHE vs. Plaintext Operations

In this experiment, we compare the performance of FHE vs. plaintext operations.

## Scenario 1: Absolute Mean Error

Aggregated from the previous experiments, the absolute mean error will be calculated to determine the accuracy of the similarity scores using FHE library.

In [6]:
mean_error_md_table(kld_err, bhatt_err, cram_err)

Function | Mean Error
---|---
KLD | 1.71e-11
Cramer | 1.61e-09
BC | 3.85e-10

## Scenario 2: Android vs. Linux


In [7]:
operations_by_sys_md_table(ops_by_sys)

System | Encryption (ms) | Decryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---|---
pc | 299.50 | 85.75 | 123.75 | 0.40
s9 | 1915.00 | 547.75 | 792.25 | 1.94
pxl | 2143.25 | 627.50 | 922.00 | 2.02

In [8]:
operations_by_sys_alg_md_table(ops_by_sys_alg)

System [Algorithm] | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---
pc [kld] | 152.25 | 49.00 | 0.26
pc [bhattacharyya] | 73.75 | 31.00 | 0.06
pc [cramer] | 73.50 | 43.75 | 0.08
s9 [kld] | 950.25 | 315.75 | 1.00
s9 [bhattacharyya] | 478.00 | 190.00 | 0.43
s9 [cramer] | 486.75 | 286.50 | 0.51
pxl [kld] | 1087.75 | 375.00 | 1.25
pxl [bhattacharyya] | 528.00 | 214.75 | 0.38
pxl [cramer] | 527.50 | 332.25 | 0.39

In [9]:
operations_by_alg_md_table(mobile_ops_by_alg)

Algorithm | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms)
---|---|---|---
kld | 1019.00 | 345.38 | 1.12
bhattacharyya | 503.00 | 202.38 | 0.41
cramer | 507.12 | 309.38 | 0.45

# Experiment 4: Pre-processing


## Scenario 1: Android vs. Linux

In [10]:
pp_by_sys

{'pc': [13.0, 74.0, 37.0, 321.0],
 's9': [180.0, 416.5, 334.5, 755.5],
 'pxl': [171.0, 418.5, 353.5, 875.0]}

In [11]:
pre_processing_by_sys_md_table(pp_by_sys)

System | Average (s) | Min (s) | Max (s)
---|---|---|---
pc | 111.25 | 13.00 | 321.00
s9 | 421.62 | 180.00 | 755.50
pxl | 454.50 | 171.00 | 875.00

## Scenario 2: Resolution

In [12]:
pp_by_res

{'720p': [121.33333333333333, 303.0], '1080p': [241.66666666666666, 650.5]}

In [13]:
pre_processing_by_res_md_table(pp_by_res)

Resolution | Average (s) | Min (s) | Max (s)
---|---|---|---
720p | 212.17 | 121.33 | 303.00
1080p | 446.08 | 241.67 | 650.50